# 🧪 Sesión 40 · Hito 4: Demostración API Full-Stack en Producción con SPA

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Tu aplicación completa resiste una demostración en vivo?

---

## 🖥️ Así se verá el resultado final

```http
GET  /health                     -> 200 {"status":"ok"}
POST /api/libros                -> 201
GET  /api/libros                -> 200
POST /api/token  (ana)          -> 200 + token
GET  /api/perfil (Bearer ana)   -> 200
GET  /api/admin/panel (ana)     -> 403
GET  /api/admin/panel (admin)   -> 200
OPTIONS /api/libros (CORS)      -> 200

RESUMEN: 8/8 casos correctos. ¡Hito 4 listo!
```

_La demo no es un video: es el sistema respondiendo en vivo, con evidencia en el repositorio._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar API, base de datos, seguridad y salud en una demo.
- Verificar el flujo completo con un solo comando.
- Presentar la evidencia en GitHub y en la nube.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El sistema está sano → HTTP 200
- [ ] Se crea un libro persistente → HTTP 201
- [ ] El libro se lista → HTTP 200
- [ ] El usuario se autentica → HTTP 200
- [ ] El usuario consulta su perfil → HTTP 200
- [ ] El usuario no entra al panel admin → HTTP 403
- [ ] El administrador sí entra → HTTP 200
- [ ] El administrador consulta el panel → HTTP 200
- [ ] CORS autoriza al cliente → HTTP 200
- [ ] Cliente SPA presente
- [ ] El cliente consume la API

## 🧩 El reto

Integra y demuestra tu API full-stack: persistencia, autenticación, roles, CORS, salud y cliente SPA.

**Requisitos del contrato**

- Usuarios de prueba: ana/secreta (user) y admin/admin123 (admin).
- La API usa SQLite y conserva los libros entre peticiones.
- Existe login con token y roles admin/user.
- El panel admin responde 200 a admin y 403 a user.
- CORS autoriza https://app.ejemplo.com.
- GET /health responde 200.
- El cliente SPA (index.html y app.js) consume la API con fetch.

> **Pista conceptual:** Para la demo, prepara datos de prueba y un guion de tres minutos: salud, crear, listar, login, perfil y panel. Si algo falla, el verificador te dice exactamente qué.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion40.md`, demo en vivo y tag `v1.0-final` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`, `index.html`, `app.js`


### 📄 `app.py`

API completa del hito final.

In [3]:
%%writefile app.py
import base64
import hashlib
import hmac
import json
import os
import secrets
import sqlite3
import time
from contextlib import closing
from pathlib import Path
from typing import Optional

from fastapi import Depends, FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field

BASE = Path(__file__).parent
DB_PATH = os.environ.get("DB_PATH", "biblioteca.db")
# Sin SECRET_KEY en el entorno, se genera una por proceso (los tokens expiran al reiniciar).
SECRET = os.environ.get("SECRET_KEY") or secrets.token_hex(32)
TOKEN_TTL = 3600  # segundos
ORIGENES = [
    o.strip()
    for o in os.environ.get(
        "CORS_ORIGINS",
        "https://app.ejemplo.com,http://localhost:8000,http://127.0.0.1:8000",
    ).split(",")
    if o.strip()
]

app = FastAPI(title="Biblioteca Full-Stack")

# CORS con lista explícita de orígenes: nunca responde con "*".
app.add_middleware(
    CORSMiddleware,
    allow_origins=ORIGENES,
    allow_methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"],
)


# ---------- Persistencia (SQLite) ----------
def conectar() -> sqlite3.Connection:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    return con


def hash_password(password: str, salt: Optional[bytes] = None) -> str:
    salt = salt or secrets.token_bytes(16)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 100_000)
    return f"{salt.hex()}${dk.hex()}"


def verificar_password(password: str, almacenado: str) -> bool:
    salt_hex, hash_hex = almacenado.split("$")
    candidato = hash_password(password, bytes.fromhex(salt_hex)).split("$")[1]
    return hmac.compare_digest(candidato, hash_hex)


def init_db() -> None:
    with closing(conectar()) as con, con:
        con.execute(
            "CREATE TABLE IF NOT EXISTS libros ("
            "id INTEGER PRIMARY KEY AUTOINCREMENT, titulo TEXT NOT NULL, anio INTEGER)"
        )
        con.execute(
            "CREATE TABLE IF NOT EXISTS usuarios ("
            "usuario TEXT PRIMARY KEY, password_hash TEXT NOT NULL, rol TEXT NOT NULL)"
        )
        for usuario, password, rol in (("ana", "secreta", "user"), ("admin", "admin123", "admin")):
            existe = con.execute("SELECT 1 FROM usuarios WHERE usuario = ?", (usuario,)).fetchone()
            if not existe:
                con.execute(
                    "INSERT INTO usuarios (usuario, password_hash, rol) VALUES (?, ?, ?)",
                    (usuario, hash_password(password), rol),
                )


init_db()


# ---------- Tokens firmados (HMAC-SHA256, solo biblioteca estándar) ----------
def _b64(datos: bytes) -> str:
    return base64.urlsafe_b64encode(datos).rstrip(b"=").decode()


def _unb64(texto: str) -> bytes:
    return base64.urlsafe_b64decode(texto + "=" * (-len(texto) % 4))


def _firmar(cuerpo: str) -> str:
    return _b64(hmac.new(SECRET.encode(), cuerpo.encode(), hashlib.sha256).digest())


def crear_token(usuario: str) -> str:
    cuerpo = _b64(json.dumps({"sub": usuario, "exp": int(time.time()) + TOKEN_TTL}).encode())
    return f"{cuerpo}.{_firmar(cuerpo)}"


def leer_token(token: str) -> Optional[str]:
    try:
        cuerpo, firma = token.split(".")
        if not hmac.compare_digest(firma, _firmar(cuerpo)):
            return None
        datos = json.loads(_unb64(cuerpo))
        if datos["exp"] < time.time():
            return None
        return datos["sub"]
    except Exception:
        return None


# ---------- Autenticación y roles ----------
bearer = HTTPBearer(auto_error=False)


def usuario_actual(cred: Optional[HTTPAuthorizationCredentials] = Depends(bearer)) -> dict:
    no_autorizado = HTTPException(
        status_code=401, detail="Token ausente o inválido", headers={"WWW-Authenticate": "Bearer"}
    )
    if cred is None:
        raise no_autorizado
    usuario = leer_token(cred.credentials)
    if usuario is None:
        raise no_autorizado
    with closing(conectar()) as con:
        fila = con.execute("SELECT usuario, rol FROM usuarios WHERE usuario = ?", (usuario,)).fetchone()
    if fila is None:
        raise no_autorizado
    return {"usuario": fila["usuario"], "rol": fila["rol"]}


def solo_admin(actual: dict = Depends(usuario_actual)) -> dict:
    if actual["rol"] != "admin":
        raise HTTPException(status_code=403, detail="Se requiere rol de administrador")
    return actual


# ---------- Modelos ----------
class LibroIn(BaseModel):
    titulo: str = Field(..., min_length=1, max_length=200)
    anio: Optional[int] = Field(None, ge=0, le=2100)


class LoginIn(BaseModel):
    usuario: str
    password: str


# ---------- Rutas ----------
@app.get("/health")
def health():
    try:
        with closing(conectar()) as con:
            con.execute("SELECT 1")
        return {"status": "ok", "db": "ok"}
    except Exception:
        return JSONResponse(status_code=503, content={"status": "error", "db": "error"})


@app.post("/api/libros", status_code=201)
def crear_libro(libro: LibroIn):
    with closing(conectar()) as con, con:
        cur = con.execute("INSERT INTO libros (titulo, anio) VALUES (?, ?)", (libro.titulo, libro.anio))
        nuevo_id = cur.lastrowid
    return {"id": nuevo_id, "titulo": libro.titulo, "anio": libro.anio}


@app.get("/api/libros")
def listar_libros():
    with closing(conectar()) as con:
        filas = con.execute("SELECT id, titulo, anio FROM libros ORDER BY id").fetchall()
    return {"libros": [dict(f) for f in filas]}


@app.post("/api/token")
def login(datos: LoginIn):
    with closing(conectar()) as con:
        fila = con.execute(
            "SELECT password_hash FROM usuarios WHERE usuario = ?", (datos.usuario,)
        ).fetchone()
    if fila is None or not verificar_password(datos.password, fila["password_hash"]):
        raise HTTPException(status_code=401, detail="Credenciales inválidas")
    return {"access_token": crear_token(datos.usuario), "token_type": "bearer"}


@app.get("/api/perfil")
def perfil(actual: dict = Depends(usuario_actual)):
    return actual


@app.get("/api/admin/panel")
def panel_admin(actual: dict = Depends(solo_admin)):
    with closing(conectar()) as con:
        total = con.execute("SELECT COUNT(*) FROM libros").fetchone()[0]
    return {"administrador": actual["usuario"], "total_libros": total}


# ---------- Cliente (solo estos dos archivos, para no exponer app.py ni la base) ----------
@app.get("/", include_in_schema=False)
def pagina():
    return FileResponse(BASE / "index.html")


@app.get("/app.js", include_in_schema=False)
def script():
    return FileResponse(BASE / "app.js", media_type="application/javascript")


Overwriting app.py


### 📄 `index.html`

Cliente SPA.

In [4]:
%%writefile index.html
<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8"><title>Biblioteca Full-Stack</title></head>
<body>
  <h1>Biblioteca</h1>
  <ul id="lista"></ul>
  <script src="app.js"></script>
</body>
</html>


Writing index.html


### 📄 `app.js`

Lógica del cliente.

In [5]:
%%writefile app.js
// Cliente de la Biblioteca: consume la API con fetch y renderiza la lista.
const API = location.protocol.startsWith("http") ? "" : "http://127.0.0.1:8000";
let token = sessionStorage.getItem("token");

function authHeaders() {
  const headers = { "Content-Type": "application/json" };
  if (token) headers["Authorization"] = "Bearer " + token;
  return headers;
}

async function login(usuario, password) {
  const res = await fetch(`${API}/api/token`, {
    method: "POST",
    headers: authHeaders(),
    body: JSON.stringify({ usuario, password }),
  });
  if (!res.ok) throw new Error("Credenciales inválidas");
  const data = await res.json();
  token = data.access_token;
  sessionStorage.setItem("token", token);
  return data;
}

async function cargarLibros() {
  const lista = document.getElementById("lista");
  const res = await fetch(`${API}/api/libros`, { headers: authHeaders() });
  if (!res.ok) throw new Error("HTTP " + res.status);
  const data = await res.json();
  lista.replaceChildren();
  for (const libro of data.libros) {
    const li = document.createElement("li");
    // textContent evita inyectar HTML si un título trae etiquetas
    li.textContent = libro.anio ? `${libro.titulo} (${libro.anio})` : libro.titulo;
    lista.appendChild(li);
  }
}

async function crearLibro(titulo, anio) {
  const res = await fetch(`${API}/api/libros`, {
    method: "POST",
    headers: authHeaders(),
    body: JSON.stringify({ titulo, anio }),
  });
  if (!res.ok) throw new Error("HTTP " + res.status);
  const libro = await res.json();
  await cargarLibros();
  return libro;
}

cargarLibros().catch((e) => {
  document.getElementById("lista").textContent = "Error al cargar libros: " + e.message;
});

Writing app.js


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [6]:
from verificar import probar_http

casos = [{'descripcion': 'El sistema está sano',
  'metodo': 'GET',
  'ruta': '/health',
  'estado': 200,
  'json_contiene': {'status': 'ok'}},
 {'descripcion': 'Se crea un libro persistente',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'El libro se lista',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'El usuario se autentica',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El usuario consulta su perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200},
 {'descripcion': 'El usuario no entra al panel admin',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'El administrador sí entra',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El administrador consulta el panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200},
 {'descripcion': 'CORS autoriza al cliente',
  'metodo': 'OPTIONS',
  'ruta': '/api/libros',
  'headers': {'Origin': 'https://app.ejemplo.com', 'Access-Control-Request-Method': 'POST'},
  'estado': 200,
  'headers_contiene': {'access-control-allow-origin': 'https://app.ejemplo.com'},
  'headers_no_contiene': {'access-control-allow-origin': '*'}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 9 casos · app.py
✅ 01. El sistema está sano
✅ 02. Se crea un libro persistente
✅ 03. El libro se lista
✅ 04. El usuario se autentica
✅ 05. El usuario consulta su perfil
✅ 06. El usuario no entra al panel admin
✅ 07. El administrador sí entra
✅ 08. El administrador consulta el panel
✅ 09. CORS autoriza al cliente
------------------------------------------------------------------------------
RESULTADO: 9/9 casos correctos. ¡Resultado logrado!


In [7]:
from verificar import revisar_archivo, resumen
resultados = []
resultados.append(revisar_archivo("index.html", exigir=["id=\"lista\"", "app.js"], descripcion="Cliente SPA presente"))
resultados.append(revisar_archivo("app.js", regex=["fetch\\(", "await\\s+.*\\.json\\(\\)", "Authorization"], descripcion="El cliente consume la API"))
resumen(resultados)


✅ Cliente SPA presente
✅ El cliente consume la API
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion40.md` a `auditorias/auditoria_sesion40.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
